In [2]:
import numpy as np
from embedder import Embedder

embedder = Embedder()
print("Embedder loaded.")

Embedder loaded.


In [4]:
query_q1 = 'How does approximate nearest neighbor search work?'

v = embedder.encode(query_q1)

print(f"Vector length : {len(v)}")
print(f"v[0]          : {v[0]:.4f}")

Vector length : 384
v[0]          : -0.0206


In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [8]:
target_file = '02-vector-search/lessons/07-sqlitesearch-vector.md'

target_doc = next(d for d in documents if d["filename"] == target_file)

v_doc = embedder.encode(target_doc["content"])

similarity = float(np.dot(v, v_doc))
print(f"Cosine similarity between query and document: {similarity:.4f}")

Cosine similarity between query and document: 0.3611


In [10]:
from gitsource import chunk_documents
from tqdm import tqdm

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")


Total chunks: 295


In [11]:
contents = [c["content"] for c in chunks]
embeddings = embedder.encode_batch(contents)
X = np.array(embeddings)
print(f"Embedding matrix shape: {X.shape}")

Embedding matrix shape: (295, 384)


In [13]:
scores = X.dot(v)

best_idx = int(np.argmax(scores))
best_chunk = chunks[best_idx]

print(f"Best score : {scores[best_idx]:.4f}")
print(f"Filename : {best_chunk['filename']}")

Best score : 0.6489
Filename : 02-vector-search/lessons/07-sqlitesearch-vector.md


In [19]:
vs.fit?

Signature: vs.fit(vectors, payload)
Docstring:
Fits the index with the provided vectors and payload documents.

Args:
    vectors (np.ndarray): 2D numpy array of shape (n_docs, vector_dimension).
    payload (list of dict): List of documents to use as payload. Each document is a dictionary.
File:      /workspaces/llm-zoomcamp/.venv/lib/python3.12/site-packages/minsearch/vector.py
Type:      method

In [26]:
from minsearch import VectorSearch

vs = VectorSearch(keyword_fields=["filename", "start"])
vs.fit(X, chunks)

query_q4 = "What metric do we use to evaluate a search engine?"
v_q4 = embedder.encode(query_q4)

results_q4 = vs.search(v_q4, num_results=5)

print("Top-5 results:")
for i, result in enumerate(results_q4):
    print(f"{i+1}. Filename: {result['filename']}, Start: {result['start']}")

Top-5 results:
1. Filename: 04-evaluation/lessons/05-search-metrics.md, Start: 0
2. Filename: 04-evaluation/lessons/01-intro.md, Start: 0
3. Filename: 01-agentic-rag/lessons/05-search.md, Start: 0
4. Filename: 04-evaluation/lessons/01-intro.md, Start: 2000
5. Filename: 04-evaluation/lessons/15-next-steps.md, Start: 0


In [33]:
from minsearch import Index

text_index = Index(
    text_fields=['content'],
    keyword_fields=['filename', 'start']
)

text_index.fit(chunks)
print("Keyword index built.")

query_q5 = 'How do I store vectors in PostgreSQL?'
v_q5 = embedder.encode(query_q5)

vec_results = vs.search(v_q5, num_results=5)
vec_filenames = {r["filename"] for r in vec_results}

text_results = text_index.search(query_q5, num_results=5)
text_filenames = {r["filename"] for r in text_results}

print("Vector search filenames:")
for f in vec_filenames:
    print(f"    {f}")

print("Text search filenames:")
for f in text_filenames:
    print(f"    {f}")

print("\nIn vector but NOT in text:")
for f in vec_filenames - text_filenames:
    print(f"    {f}")    

Keyword index built.
Vector search filenames:
    03-orchestration/lessons/05-rag.md
    02-vector-search/lessons/08-pgvector.md
Text search filenames:
    02-vector-search/lessons/02-embeddings.md
    03-orchestration/lessons/05-rag.md
    02-vector-search/lessons/01-intro.md

In vector but NOT in text:
    02-vector-search/lessons/08-pgvector.md


In [35]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query_q6 = "How do I give the model access to tools?"
v_q6 = embedder.encode(query_q6)

vec_results_q6 = vs.search(v_q6, num_results=5)
text_results_q6 = text_index.search(query_q6, num_results=5)

hybrid_results_q6 = rrf([vec_results_q6, text_results_q6])

print("Hybrid search (RRF) top-5:")
for i,r in enumerate(hybrid_results_q6):
    print(f"    {i+1}. Filename: {r['filename']}, Start: {r['start']}")

print(f"Q6 answer: {hybrid_results_q6[0]['filename']}")

Hybrid search (RRF) top-5:
    1. Filename: 01-agentic-rag/lessons/13-function-calling.md, Start: 4000
    2. Filename: 01-agentic-rag/lessons/01-intro.md, Start: 2000
    3. Filename: 01-agentic-rag/lessons/14-agentic-loop.md, Start: 0
    4. Filename: 04-evaluation/lessons/02-ground-truth.md, Start: 1000
    5. Filename: 01-agentic-rag/lessons/16-other-frameworks.md, Start: 0
Q6 answer: 01-agentic-rag/lessons/13-function-calling.md
